# Week 5, Notebook 2: Convolutions and Filters

**The trick that lets a computer find edges, stripes, and shapes.**

*Caribbean AI - Adrian Dunkley*

---

### What you will build
- A **convolution** done by hand, so you see exactly what it does.
- Real **filters** that find vertical edges, horizontal edges, and blur an image.
- The link between these filters and the "Conv" layers inside a real CNN.

### The big idea
Last notebook, an image became a grid of numbers. But how does a network spot a
**stripe** on the Barbados flag or the **spiral** of a hurricane? It slides a
tiny window, called a **filter** (or **kernel**), across the image and checks how
well each patch matches a small pattern. This sliding-and-matching is a
**convolution**.

A CNN (Convolutional Neural Network) is just a network that **learns its own
filters** instead of us picking them. To understand a CNN, you first watch one
filter work. Let us do that by hand.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# A simple image: a bright vertical stripe down the middle (like a flag band).
img = np.zeros((9, 9))
img[:, 4] = 1.0
img[:, 3] = 0.6
img[:, 5] = 0.6

plt.figure(figsize=(3, 3))
plt.imshow(img, cmap="gray"); plt.title("A vertical stripe"); plt.axis("off")
plt.show()
print(img)

### Step 1: What a filter is

A filter is a small grid of numbers, often 3x3. To use it, you lay it over a 3x3
patch of the image, **multiply matching cells, and add up the results** into a
single number. That number says "how strongly this patch matches the filter".
Then you slide the filter one pixel over and repeat, across the whole image.

Here is a filter that detects a **vertical edge**: bright where the image goes
from dark on the left to bright on the right.

In [ ]:
# A classic vertical-edge filter (called a Sobel filter).
vertical_edge = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1],
])
print("Vertical-edge filter:\n", vertical_edge)

### Step 2: Do the convolution by hand

We slide the 3x3 filter over every position of the image. At each stop we
multiply and sum. The result is a new grid, the **feature map**, that lights up
wherever the filter's pattern appears. We write the loop out in full so nothing
is hidden.

In [ ]:
def convolve(image, kernel):
    kh, kw = kernel.shape
    h, w = image.shape
    out = np.zeros((h - kh + 1, w - kw + 1))
    # slide the kernel over every valid position
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            patch = image[i:i+kh, j:j+kw]      # the 3x3 patch under the filter
            out[i, j] = np.sum(patch * kernel) # multiply matching cells, add up
    return out

feature_map = convolve(img, vertical_edge)
print("Feature map shape:", feature_map.shape)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.3))
axes[0].imshow(img, cmap="gray"); axes[0].set_title("input: a stripe")
axes[1].imshow(feature_map, cmap="gray"); axes[1].set_title("after vertical-edge filter")
for a in axes: a.axis("off")
plt.show()

See how the filter lit up along the **edges** of the stripe, where dark meets
bright, and stayed flat in the plain areas? That is edge detection, done with
nothing but multiply-and-add.

### Step 3: Different filters find different things

Swap the filter and you find different patterns. A **horizontal-edge** filter is
the vertical one turned on its side. A **blur** filter averages each patch,
smoothing the image. Let us try all three on a real Caribbean flag.

In [ ]:
# Load a flag and turn it into grayscale so we can see edges clearly.
data = np.load("data/caribbean_flags.npz", allow_pickle=True)
names = list(data["class_names"])
idx = np.where(data["y"] == names.index("Barbados"))[0][0]
colour = data["X"][idx].astype("float32") / 255.0
gray = colour.mean(axis=2)          # average the 3 channels -> grayscale

horizontal_edge = vertical_edge.T   # transpose: rows become columns
blur = np.ones((3, 3)) / 9.0        # average of the 9 pixels

maps = {
    "grayscale flag": gray,
    "vertical edges": convolve(gray, vertical_edge),
    "horizontal edges": convolve(gray, horizontal_edge),
    "blurred": convolve(gray, blur),
}
fig, axes = plt.subplots(1, 4, figsize=(13, 3.3))
for ax, (title, m) in zip(axes, maps.items()):
    ax.imshow(m, cmap="gray"); ax.set_title(title); ax.axis("off")
plt.suptitle("One image, four filters (Barbados flag)")
plt.show()

The **vertical-edge** filter found the boundaries between the Barbados flag's
blue and gold vertical bands, while the **horizontal-edge** filter found almost
nothing there (the bands have no horizontal edges). A network can tell flags
apart precisely because different flags fire different filters.

### Step 4: Stacking filters is what "deep" means for vision

A CNN puts many filters in a layer, then **stacks layers**. Early layers find
simple things (edges, corners). Later layers combine those into bigger ideas (a
stripe, a triangle, a spiral). You do not design the filters; the network
**learns** them during training, using the very same gradient descent from Week 4.

Let us confirm the pattern-matching idea one more way: run the vertical-edge
filter on a **hurricane** tile and watch it trace the swirling cloud bands.

In [ ]:
sat = np.load("data/satellite_tiles.npz", allow_pickle=True)
snames = list(sat["class_names"])
hidx = np.where(sat["y"] == snames.index("hurricane"))[0][0]
hurricane = sat["X"][hidx, :, :, 0].astype("float32") / 255.0

fig, axes = plt.subplots(1, 2, figsize=(7, 3.3))
axes[0].imshow(hurricane, cmap="gray"); axes[0].set_title("hurricane tile")
axes[1].imshow(np.abs(convolve(hurricane, vertical_edge)), cmap="magma")
axes[1].set_title("edges the filter finds")
for a in axes: a.axis("off")
plt.show()

### What you learned

- A **filter** (kernel) is a small grid of numbers. A **convolution** slides it
  over an image, multiplying and adding, to make a **feature map**.
- Different filters detect different things: **vertical edges**, **horizontal
  edges**, **blur**, and much more.
- A **CNN** stacks layers of filters and **learns** them from data, building up
  from edges to shapes. That is why it can recognise flags and hurricanes.

Next notebook we stop hand-picking filters and let **Keras** build and train a
real CNN to classify six Caribbean flags.

### Try it yourself
1. Invent your own 3x3 filter (any numbers) and see what it highlights.
2. Run all four filters on a **Jamaica** flag (lots of diagonals). Which filter
   reacts most?
3. Make a diagonal-edge filter and test it on the Trinidad and Tobago flag.